In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [3]:
from src.data_loader import load_data
df = load_data()

df.head()

,Booking_ID,number of adults,number of children,number of weekend nights,number of week nights,type of meal,car parking space,room type,lead time,market segment type,repeated,P-C,P-not-C,average price,special requests,date of reservation,booking status
0,INN00001,1,1,2,5,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,88.00,0,10/2/2015,Not_Canceled
1,INN00002,1,0,1,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,11/6/2018,Not_Canceled
2,INN00003,2,1,1,3,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,50.00,0,2/28/2018,Canceled
3,INN00004,1,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,1,5/20/2017,Canceled
4,INN00005,1,0,1,2,Not Selected,0,Room_Type 1,48,Online,0,0,0,77.00,0,4/11/2018,Canceled


In [5]:
# Features
X = df.drop(columns=["Booking_ID" , "date of reservation" , "booking status"])

# Target
y = df["booking status"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)

Features:
['number of adults', 'number of children', 'number of weekend nights', 'number of week nights', 'type of meal', 'car parking space', 'room type', 'lead time', 'market segment type', 'repeated', 'P-C', 'P-not-C', 'average price', 'special requests']

Target:
booking status

Features Shape: (36285, 14)
Target Shape: (36285,)


In [10]:
# Numerical Features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Categorical Features
categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Numerical Features:", numerical_features)
print("Number of Numerical Features:", len(numerical_features))

print("\nCategorical Features:", categorical_features)
print("Number of Categorical Features:", len(categorical_features))

Numerical Features: ['number of adults', 'number of children', 'number of weekend nights', 'number of week nights', 'car parking space', 'lead time', 'repeated', 'P-C', 'P-not-C', 'average price', 'special requests']
Number of Numerical Features: 11

Categorical Features: ['type of meal', 'room type', 'market segment type']
Number of Categorical Features: 3


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Features Shape: (29028, 14)
Testing Features Shape: (7257, 14)

Training Target Shape: (29028,)
Testing Target Shape: (7257,)

Training Target Distribution:
booking status
Not_Canceled    19517
Canceled         9511
Name: count, dtype: int64

Testing Target Distribution:
booking status
Not_Canceled    4879
Canceled        2378
Name: count, dtype: int64


In [9]:
print("\nTotal Missing Values in Training Set:")
print(X_train.isnull().sum().sum())

print("\nTotal Missing Values in Testing Set:")
print(X_test.isnull().sum().sum())


Total Missing Values in Training Set:
0

Total Missing Values in Testing Set:
0


In [11]:
for feature in categorical_features:
    print(f"{feature}:")
    print(X_train[feature].unique())
    print("-" * 40)

type of meal:
<ArrowStringArray>
['Meal Plan 1', 'Meal Plan 2', 'Not Selected', 'Meal Plan 3']
Length: 4, dtype: str
----------------------------------------
room type:
<ArrowStringArray>
['Room_Type 1', 'Room_Type 4', 'Room_Type 5', 'Room_Type 6', 'Room_Type 2',
 'Room_Type 7', 'Room_Type 3']
Length: 7, dtype: str
----------------------------------------
market segment type:
<ArrowStringArray>
['Offline', 'Online', 'Complementary', 'Corporate', 'Aviation']
Length: 5, dtype: str
----------------------------------------


In [12]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [13]:
X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

In [14]:
print("Encoded Data Shape:", X_train_encoded.shape)

print("\nEncoded Feature Names:")
print(encoder.get_feature_names_out(categorical_features))

Encoded Data Shape: (29028, 16)

Encoded Feature Names:
['type of meal_Meal Plan 1' 'type of meal_Meal Plan 2'
 'type of meal_Meal Plan 3' 'type of meal_Not Selected'
 'room type_Room_Type 1' 'room type_Room_Type 2' 'room type_Room_Type 3'
 'room type_Room_Type 4' 'room type_Room_Type 5' 'room type_Room_Type 6'
 'room type_Room_Type 7' 'market segment type_Aviation'
 'market segment type_Complementary' 'market segment type_Corporate'
 'market segment type_Offline' 'market segment type_Online']


In [15]:
X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

In [16]:
print("Training Encoded Shape:", X_train_encoded.shape)
print("Testing Encoded Shape:", X_test_encoded.shape)

Training Encoded Shape: (29028, 16)
Testing Encoded Shape: (7257, 16)


In [17]:

encoded_feature_names = encoder.get_feature_names_out()

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

In [18]:
X_train_processed = pd.concat(
    [
        X_train[numerical_features],
        X_train_encoded_df
    ],
    axis=1
)

X_test_processed = pd.concat(
    [
        X_test[numerical_features],
        X_test_encoded_df
    ],
    axis=1
)

In [19]:
X_train_processed

,number of adults,number of children,number of weekend nights,number of week nights,car parking space,lead time,repeated,P-C,P-not-C,average price,...,room type_Room_Type 3,room type_Room_Type 4,room type_Room_Type 5,room type_Room_Type 6,room type_Room_Type 7,market segment type_Aviation,market segment type_Complementary,market segment type_Corporate,market segment type_Offline,market segment type_Online
12794,2,0,0,2,0,322,0,0,0,52.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1439,2,0,2,2,0,54,0,0,0,81.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
35466,3,0,2,3,0,237,0,0,0,104.55,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3462,1,0,2,1,0,0,1,0,5,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
21306,2,0,0,1,0,62,0,0,0,159.30,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9270,2,0,0,3,0,304,0,0,0,89.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4449,1,0,1,0,1,0,1,0,1,88.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
28672,1,0,0,1,0,0,1,0,3,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
30662,2,2,2,2,0,130,0,0,0,144.50,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [20]:
print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

print("\nTraining Data Types:")
print(X_train_processed.dtypes.value_counts())

print("\nFirst 5 Rows:")
display(X_train_processed.head())

Training Data Shape: (29028, 27)
Testing Data Shape: (7257, 27)

Training Data Types:
float64    17
int64      10
Name: count, dtype: int64

First 5 Rows:


,number of adults,number of children,number of weekend nights,number of week nights,car parking space,lead time,repeated,P-C,P-not-C,average price,...,room type_Room_Type 3,room type_Room_Type 4,room type_Room_Type 5,room type_Room_Type 6,room type_Room_Type 7,market segment type_Aviation,market segment type_Complementary,market segment type_Corporate,market segment type_Offline,market segment type_Online
12794,2,0,0,2,0,322,0,0,0,52.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1439,2,0,2,2,0,54,0,0,0,81.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
35466,3,0,2,3,0,237,0,0,0,104.55,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3462,1,0,2,1,0,0,1,0,5,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
21306,2,0,0,1,0,62,0,0,0,159.30,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [21]:
X_train[numerical_features].describe().T

,count,mean,std,min,25%,50%,75%,max
number of adults,29028.0,1.846872,0.520114,0.0,2.00,2.0,2.00,4.0
number of children,29028.0,0.105415,0.403307,0.0,0.00,0.0,0.00,10.0
number of weekend nights,29028.0,0.811630,0.871374,0.0,0.00,1.0,2.00,7.0
number of week nights,29028.0,2.202632,1.412216,0.0,1.00,2.0,3.00,17.0
car parking space,29028.0,0.031315,0.174170,0.0,0.00,0.0,0.00,1.0
lead time,29028.0,85.228366,86.006877,0.0,17.00,57.0,126.00,443.0
repeated,29028.0,0.025389,0.157307,0.0,0.00,0.0,0.00,1.0
P-C,29028.0,0.021428,0.350731,0.0,0.00,0.0,0.00,13.0
P-not-C,29028.0,0.152818,1.757435,0.0,0.00,0.0,0.00,58.0
average price,29028.0,103.565938,35.148189,0.0,80.75,99.9,120.69,540.0


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [23]:
X_train_scaled = scaler.fit_transform(
    X_train[numerical_features]
)

In [24]:
print("Scaled Data Shape:", X_train_scaled.shape)

print("\nScaled Data Mean:")
print(X_train_scaled.mean(axis=0))

print("\nScaled Data Standard Deviation:")
print(X_train_scaled.std(axis=0))

Scaled Data Shape: (29028, 11)

Scaled Data Mean:
[ 1.27284767e-17 -7.93082012e-17 -3.91645438e-18  8.56724395e-18
  3.62272030e-17  3.18211918e-18 -1.17493631e-17 -1.51762607e-17
  9.79113595e-19 -1.49804380e-16  6.07050429e-17]

Scaled Data Standard Deviation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [25]:
X_test_scaled = scaler.transform(
    X_test[numerical_features]
)

In [26]:
print("Training Scaled Shape:", X_train_scaled.shape)
print("Testing Scaled Shape:", X_test_scaled.shape)

Training Scaled Shape: (29028, 11)
Testing Scaled Shape: (7257, 11)


In [27]:
# Convert scaled numerical features into DataFrames
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=numerical_features,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=numerical_features,
    index=X_test.index
)

# Combine numerical and categorical features
X_train_processed = pd.concat(
    [
        X_train_scaled_df,
        X_train_encoded_df
    ],
    axis=1
)

X_test_processed = pd.concat(
    [
        X_test_scaled_df,
        X_test_encoded_df
    ],
    axis=1
)

print("Training Processed Shape:", X_train_processed.shape)
print("Testing Processed Shape:", X_test_processed.shape)

Training Processed Shape: (29028, 27)
Testing Processed Shape: (7257, 27)


In [30]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

In [31]:
print("Target Classes:", target_encoder.classes_)

print("\nEncoded Training Target:")
print(pd.Series(y_train_encoded).value_counts())

print("\nEncoded Testing Target:")
print(pd.Series(y_test_encoded).value_counts())

Target Classes: ['Canceled' 'Not_Canceled']

Encoded Training Target:
1    19517
0     9511
Name: count, dtype: int64

Encoded Testing Target:
1    4879
0    2378
Name: count, dtype: int64


In [38]:
# Check final dataset shapes
print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

# Check data types
print("\nData Types:")
print(X_train_processed.dtypes.value_counts())

# Check missing values
print("\nTraining Missing Values:", X_train_processed.isnull().sum().sum())
print("Testing Missing Values:", X_test_processed.isnull().sum().sum())

# Check index alignment
print("\nTraining Index Alignment:", X_train_processed.index.equals(y_train.index))
print("Testing Index Alignment:", X_test_processed.index.equals(y_test.index))

# Check target classes
print("\nTarget Classes: \n", pd.Series(y_train_encoded).value_counts())

Training Data Shape: (29028, 27)
Testing Data Shape: (7257, 27)

Data Types:
float64    27
Name: count, dtype: int64

Training Missing Values: 0
Testing Missing Values: 0

Training Index Alignment: True
Testing Index Alignment: True

Target Classes: 
 1    19517
0     9511
Name: count, dtype: int64
